## Import the necessary functions 

In [1]:
import xarray as xr
import numpy as np
import matplotlib.pyplot as plt
from datetime import datetime
import time

## Edit the domain CFG file to the new grid outline

In [2]:
# Set the filepaths that you might need 

# CFG file that you would like to modify
#filepath_old_cfg = '/bettik/ockendeh/NEMO_simulations/TGCC/eORCA1.4.2_domain_cfg.nc'
filepath_old_cfg = '/bettik/ockendeh/NEMO_simulations/TGCC/eORCA1.4.3_OpenSeas_OpenAllCav_ModStraights_domain_cfg.nc'

# Mask with the outline of the ocean region for the NN
filepath_eORCA1 = '/bettik/ockendeh/NEMO_simulations/eORCA1_bmach_masks.nc'

# OUTPUT filepath for the modified CFG
filepath_new_cfg = '/bettik/ockendeh/NEMO_simulations/TGCC/eORCA1.4.3_CavsForNN_domain_cfg.nc'

# Do you want to plot the changes to the domain CFG file as you go along?
plot_changes = False

In [3]:
# Load in the OLD CFG file that needs to be altered 
CFG = xr.open_dataset(filepath_old_cfg)

# Load in the mask of the ocean region
mask_eORCA1 =  xr.open_dataset(filepath_eORCA1)
ocean_mask = mask_eORCA1.mask_nocavs
mask_eORCA1.close()

In [4]:
## What we want to do is to crop the top and bottom levels 
## They should have a value >0 in the ocean region
## They should be 0 everywhere else 

In [5]:
# Redo the top level 

# Copy the top level from the CFG file 
top_level = CFG.top_level.isel(t=0).squeeze().copy().values
# Change to float so you can set the non ocean regions to nan
top_level = top_level.astype('float32')
top_level[top_level == 0] = np.nan

# Because the ocean mask is not the same shape as the cfg file 
# Set the bits that are outside the Antarctic domain to be the same as the CFG file 
ocean_mask[97:,:] = top_level[97:110,:]

# Set the parts that are in the Antarctic domain using the ocean mask 
top_level[:110,:] = top_level[:110,:] * ocean_mask
# Change nan regions back to 0
top_level[np.isnan(top_level)] = 0
# Change array type back to int
top_level = top_level.astype('int32')

In [6]:
if plot_changes == True:
    fig, ax = plt.subplots(1,3,figsize = (15,5))
    plt.subplots_adjust(wspace = 0.02)
    im = [[],[],[]]
    im[0] = ax[0].pcolor(CFG.top_level.isel(t=0).squeeze().copy().values[:110,:])
    im[1] = ax[1].pcolor(top_level[:110,:])
    im[2] = ax[2].pcolor(CFG.top_level.isel(t=0).squeeze().copy().values[:110,:] - top_level[:110,:], cmap = 'coolwarm')
    #plt.colorbar()
    titles = 'Top level\nOld CFG', 'Top level\nNew CFG', 'Difference'
    for i in range(3):
        ax[i].set_xticks([])
        ax[i].set_yticks([])
        ax[i].set_title(titles[i], fontweight = 'bold', fontsize = 12)

In [7]:
# Redo the bottom level 

# Copy the bottom level from the CFG file 
bot_level = CFG.bottom_level.isel(t=0).squeeze().copy().values
# Change to float so you can set the non ocean regions to nan
bot_level = bot_level.astype('float32')
bot_level[bot_level == 0] = np.nan

# Set the parts that are in the Antarctic domain using the ocean mask 
bot_level[:110,:] = bot_level[:110,:] * ocean_mask
# Change nan regions back to 0
bot_level[np.isnan(bot_level)] = 0
# Change array type back to int
bot_level = bot_level.astype('int32')

In [8]:
if plot_changes == True:
    fig, ax = plt.subplots(1,3,figsize = (15,5))
    plt.subplots_adjust(wspace = 0.02)
    im = [[],[],[]]
    im[0] = ax[0].pcolor(CFG.bottom_level.isel(t=0).squeeze().copy().values[:110,:])
    im[1] = ax[1].pcolor(bot_level[:110,:])
    im[2] = ax[2].pcolor(CFG.bottom_level.isel(t=0).squeeze().copy().values[:110,:] - bot_level[:110,:], cmap = 'coolwarm')
    #plt.colorbar()
    titles = 'Bottom level\nOld CFG', 'Bottom level\nNew CFG', 'Difference'
    for i in range(3):
        ax[i].set_xticks([])
        ax[i].set_yticks([])
        ax[i].set_title(titles[i], fontweight = 'bold', fontsize = 12)

In [9]:
# Redo the bathymetry

# Copy the bottom level from the CFG file 
bathy = CFG.bathy_metry.isel(t=0).squeeze().copy().values
# Set the non ocean regions to nan
bathy[bathy == 0] = np.nan

# Set the parts that are in the Antarctic domain using the ocean mask 
bathy[:110,:] = bathy[:110,:] * ocean_mask
# Change nan regions back to 0
bathy[np.isnan(bathy)] = 0

In [10]:
if plot_changes == True:
    fig, ax = plt.subplots(1,3,figsize = (15,5))
    plt.subplots_adjust(wspace = 0.02)
    im = [[],[],[]]
    im[0] = ax[0].pcolor(CFG.bathy_metry.isel(t=0).squeeze().copy().values[:110,:])
    im[1] = ax[1].pcolor(bathy[:110,:])
    im[2] = ax[2].pcolor(CFG.bathy_metry.isel(t=0).squeeze().copy().values[:110,:] - bathy[:110,:], cmap = 'coolwarm')
    #plt.colorbar()
    titles = 'Bathymetry\nOld CFG', 'Bathymetry\nNew CFG', 'Difference'
    for i in range(3):
        ax[i].set_xticks([])
        ax[i].set_yticks([])
        ax[i].set_title(titles[i], fontweight = 'bold', fontsize = 12)

In [11]:
# Redo the ice shelf draft

# Copy the bottom level from the CFG file 
draft = CFG.isf_draft.isel(t=0).squeeze().copy().values
# Set the non ocean regions to nan
draft[draft == 0] = np.nan

# Set the parts that are in the Antarctic domain using the ocean mask 
draft[:110,:] = draft[:110,:] * ocean_mask
# Change nan regions back to 0
draft[np.isnan(draft)] = 0

In [12]:
if plot_changes == True:
    fig, ax = plt.subplots(1,3,figsize = (15,5))
    plt.subplots_adjust(wspace = 0.02)
    im = [[],[],[]]
    im[0] = ax[0].pcolor(CFG.isf_draft.isel(t=0).squeeze().copy().values[:110,:])
    im[1] = ax[1].pcolor(draft[:110,:])
    im[2] = ax[2].pcolor(CFG.isf_draft.isel(t=0).squeeze().copy().values[:110,:] - draft[:110,:], cmap = 'coolwarm')
    #plt.colorbar()
    titles = 'Ice Shelf draft\nOld CFG', 'Ice Shelf draft\nNew CFG', 'Difference'
    for i in range(3):
        ax[i].set_xticks([])
        ax[i].set_yticks([])
        ax[i].set_title(titles[i], fontweight = 'bold', fontsize = 12)

In [13]:
# Create a new xr dataset with the changed arrays 
changes = xr.Dataset({
                      "top_level":    (("t", "y", "x"), top_level[np.newaxis, :, :]),
                      "bottom_level": (("t", "y", "x"), bot_level[np.newaxis, :, :]),
                      "bathy_metry":   (("t", "y", "x"), bathy[np.newaxis, :, :]),
                      "isf_draft":    (("t", "y", "x"), draft[np.newaxis, :, :]),
                     })
#changes = changes.drop_vars(["t", "y", "x"])
New_CFG = CFG.merge(changes, overwrite_vars=list(changes.data_vars))

In [14]:
# Check that the new CFG file has the same structure as the old CFG file
# Note that this is not a particularly robust checking function

def assert_same_structure(ds1, ds2):
    assert ds1.sizes == ds2.sizes
    assert set(ds1.data_vars) == set(ds2.data_vars)
    for v in ds1.data_vars:
        assert ds1[v].dtype == ds2[v].dtype
        assert ds1[v].shape == ds2[v].shape
    # Check that the two arrays have the same coordinates
    for v in ds1.coords:
        assert ds1[v] == ds2[v]
    for v in ds2.coords:
        assert ds2[v] == ds1[v]
        
try:
    assert_same_structure(CFG,New_CFG)
    print('New array has the same basic structure')
except:
    print('There are errors, but ignore differing data variables')
    xr.testing.assert_equal(CFG, New_CFG)

New array has the same basic structure


In [15]:
## Update the Attributes of the CFG file 

# Update the timestamp
New_CFG.attrs["TimeStamp"] = datetime.now().strftime("%d/%m/%Y %H:%M:%S %z")

# Update the history 
CFG_history = New_CFG.attrs.get("history", "")
update = f"{time.ctime()}: Modified top/bottom/bathy/draft to allow partial cells to be resolved using neural network. " + \
            f"{filepath_new_cfg.split('/')[-1]}"

New_CFG.attrs["history"] = f"{CFG_history}\n{update}".strip()

In [16]:
New_CFG

<xarray.Dataset> Size: 526MB
Dimensions:           (t: 1, y: 331, x: 360, z: 75, nlines: 179)
Dimensions without coordinates: t, y, x, z, nlines
Data variables: (12/53)
    e1v               (t, y, x) float64 953kB ...
    e2u               (t, y, x) float64 953kB ...
    strait_shlat      (t, y, x) float64 953kB ...
    nav_lon           (y, x) float32 477kB ...
    nav_lat           (y, x) float32 477kB ...
    nav_lev           (z) float32 300B ...
    ...                ...
    mask_csemp        (t, y, x) int32 477kB ...
    mask_csrnf        (t, y, x) int32 477kB ...
    mask_csgrpglo     (t, y, x) int32 477kB ...
    mask_csgrpemp     (t, y, x) int32 477kB ...
    mask_csgrprnf     (t, y, x) int32 477kB ...
    namelist_dom_cfg  (nlines) |S177 32kB ...
Attributes:
    file_name:  domain_cfg.nc
    TimeStamp:  30/03/2026 12:22:52 
    NCO:        netCDF Operators version 5.0.1 (Homepage = http://nco.sf.net,...
    history:    Tue Oct  1 18:43:55 2024: ncrename -d nav_lev,z -d time_count...

In [17]:
# Save new CFG file to specified filepath
New_CFG.to_netcdf(filepath_new_cfg)
print('You have saved: ', filepath_new_cfg)

You have saved:  /bettik/ockendeh/NEMO_simulations/TGCC/eORCA1.4.3_CavsForNN_domain_cfg.nc
